In [0]:
#Qn.Create a PySpark program to analyze a table containing null values and calculate the number of null and not null values for each column

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, when

# Initialize Spark session
spark = SparkSession.builder.appName("NullCount").getOrCreate()

# sample dataframe
data = [
    ("John", 25, None),
    ("Sarah", None, "Paris"),
    (None, 30, None),
    ("Tom", 40, "London"),
    (None, None, None)
]
columns = ["name", "age", "city"]
df = spark.createDataFrame(data, columns)
df.show()

# Create a list of expressions to compute null and not null counts for each column
expressions = [
    (col_name, 
     sum(when(col(col_name).isNull(), 1).otherwise(0)).alias("nulls"),
     sum(when(col(col_name).isNotNull(), 1).otherwise(0)).alias("not_nulls"))
    for col_name in df.columns
]

# Convert the expressions into a DataFrame with the appropriate column names
result = spark.createDataFrame(
    [(col_name, 
      df.select(sum(when(col(col_name).isNull(), 1).otherwise(0))).collect()[0][0], 
      df.select(sum(when(col(col_name).isNotNull(), 1).otherwise(0))).collect()[0][0])
     for col_name in df.columns],
    ["column", "nulls", "not_nulls"]
)

# Show the result
result.show()


+-----+----+------+
| name| age|  city|
+-----+----+------+
| John|  25|  null|
|Sarah|null| Paris|
| null|  30|  null|
|  Tom|  40|London|
| null|null|  null|
+-----+----+------+

+------+-----+---------+
|column|nulls|not_nulls|
+------+-----+---------+
|  name|    2|        3|
|   age|    2|        3|
|  city|    3|        2|
+------+-----+---------+

